# ATM Cash Demand - Time Series Data Preprocessing & EDA

## Overview
This notebook handles:
1. **Data Loading**: Load NN5 COMPLETE dataset from Excel
2. **Data Transformation**: Convert to long format and handle timestamps
3. **Train/Val/Test Split**: Proper time-series split with 56-day horizons
4. **EDA (Exploratory Data Analysis)**:
   - Trend analysis with moving averages
   - Autocorrelation (ACF) and Partial Autocorrelation (PACF)
   - Weekly and monthly seasonality patterns
   - Volatility analysis
5. **Data Quality Checks**: Missing values, duplicates, data types
6. **Data Export**: Save processed datasets and EDA plots

## Output Structure
- **data/processed/**: Train, Val, Test dataframes
- **report/figures/**: 6 high-quality EDA visualizations

## 1. Import Required Libraries

In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.dates as mdates
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import warnings
import os

warnings.filterwarnings('ignore')

# Set global styles
MAIN_BLUE = '#1f497d'
LIGHT_GREY = '#d9d9d9'
TEXT_GREY = '#595959'
AXIS_GREY = '#8c8c8c'
BG_HIGHLIGHT = '#f2f2f2'

sns.set_style('white')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['text.color'] = TEXT_GREY
plt.rcParams['axes.labelcolor'] = TEXT_GREY
plt.rcParams['xtick.color'] = TEXT_GREY
plt.rcParams['ytick.color'] = TEXT_GREY

# Ensure output directories exist
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../report/figures', exist_ok=True)

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


## 2. Load Raw Data from Excel

In [40]:
# ========================= CONFIG =========================
DATA_PATH = "../data/raw/NN5_COMPLETE.xls"
SHEET_NAME = "NN3 COMPLETE Data"
SKIPROWS = 13
FORECAST_HORIZON = 56
# =======================================================

# Load raw data
df_raw = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME, skiprows=SKIPROWS)

# Extract time series data (skip first 3 rows which are metadata)
ts_data = df_raw.iloc[3:].copy()
ts_data.rename(columns={'Time Series #ID': 'Excel_Date'}, inplace=True)
ts_data['Date'] = pd.to_datetime(ts_data['Excel_Date'], errors='coerce')
ts_data.set_index('Date', inplace=True)
ts_data.drop(columns=['Excel_Date'], inplace=True)
ts_data = ts_data.astype(float)

print(f"✅ Data Loaded: {ts_data.shape[0]} days × {ts_data.shape[1]} ATMs")
print(f"\n📊 Date Range: {ts_data.index.min()} to {ts_data.index.max()}")
print(f"\n🏧 ATM Count: {ts_data.shape[1]}")
print(f"\n📅 Total Days: {ts_data.shape[0]}")

✅ Data Loaded: 791 days × 111 ATMs

📊 Date Range: 1996-03-18 00:00:00 to 1998-05-17 00:00:00

🏧 ATM Count: 111

📅 Total Days: 791


## 3. Data Transformation & Reshaping

In [41]:
# Convert wide format to long format
df_long = (ts_data.reset_index()
           .melt(id_vars='Date',
                 var_name='ATM_ID',
                 value_name='Withdrawal')
           .sort_values(['ATM_ID', 'Date'])
           .reset_index(drop=True))

print(f"✅ Data reshaped to long format")
print(f"\n📋 Shape: {df_long.shape}")
print(f"\n🔍 Sample data:")
print(df_long.head(10))

✅ Data reshaped to long format

📋 Shape: (87801, 3)

🔍 Sample data:
        Date   ATM_ID  Withdrawal
0 1996-03-18  NN5-001   13.407029
1 1996-03-19  NN5-001   14.725057
2 1996-03-20  NN5-001   20.564059
3 1996-03-21  NN5-001   34.708050
4 1996-03-22  NN5-001   26.629819
5 1996-03-23  NN5-001   16.609977
6 1996-03-24  NN5-001   15.320295
7 1996-03-25  NN5-001   11.607143
8 1996-03-26  NN5-001   19.883787
9 1996-03-27  NN5-001   23.767007


## 4. Train/Validation/Test Split

In [42]:
# ====================== CHIA DỮ LIỆU ĐÚNG ======================
# 56 ngày cuối dành cho Submission (không có nhãn)
submission_df = df_long.groupby('ATM_ID').tail(FORECAST_HORIZON).copy()

# Phần dữ liệu có nhãn
data_with_gt = df_long.groupby('ATM_ID').head(-FORECAST_HORIZON).copy()

# Chia train / val / test internal theo thứ tự thời gian
test_df  = data_with_gt.groupby('ATM_ID').tail(FORECAST_HORIZON).copy()           # Test internal

# Lấy Val: 56 ngày ngay trước Test
val_df   = data_with_gt.groupby('ATM_ID').apply(
    lambda x: x.iloc[-FORECAST_HORIZON*2 : -FORECAST_HORIZON]
).reset_index(drop=True)

# Train: phần còn lại
train_df = data_with_gt.groupby('ATM_ID').apply(
    lambda x: x.iloc[:-FORECAST_HORIZON*2]
).reset_index(drop=True)

print(f"📊 Train shape       : {train_df.shape}")
print(f"📊 Val shape         : {val_df.shape}")
print(f"📊 Test (internal)   : {test_df.shape}")
print(f"📊 Submission shape  : {submission_df.shape}")

print(f"\nTrain date     : {train_df['Date'].min()} → {train_df['Date'].max()}")
print(f"Val date       : {val_df['Date'].min()} → {val_df['Date'].max()}")
print(f"Test date      : {test_df['Date'].min()} → {test_df['Date'].max()}")
print(f"Submission date: {submission_df['Date'].min()} → {submission_df['Date'].max()}")

📊 Train shape       : (69153, 3)
📊 Val shape         : (6216, 3)
📊 Test (internal)   : (6216, 3)
📊 Submission shape  : (6216, 3)

Train date     : 1996-03-18 00:00:00 → 1997-11-30 00:00:00
Val date       : 1997-12-01 00:00:00 → 1998-01-25 00:00:00
Test date      : 1998-01-26 00:00:00 → 1998-03-22 00:00:00
Submission date: 1998-03-23 00:00:00 → 1998-05-17 00:00:00


In [43]:
import pandas as pd
import numpy as np

def create_features(df: pd.DataFrame, is_train=False):
    """
    Tạo features toàn diện: Time, UK Holidays, Lags, Rolling.
    An toàn 100% chống rò rỉ dữ liệu (No Data Leakage).
    """
    df = df.copy()
    df = df.sort_values(['ATM_ID', 'Date'])
    
    # =======================================================
    # 1. TIME & UK HOLIDAY FEATURES (Không bao giờ bị Leak)
    # =======================================================
    # Các biến thời gian cơ bản
    df['day_of_week'] = df['Date'].dt.dayofweek
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    df['day_of_month'] = df['Date'].dt.day
    df['month'] = df['Date'].dt.month
    df['is_payday'] = df['day_of_month'].isin([1, 15, 25, 28, 30, 31]).astype(int)
    
    # Danh sách Bank Holidays UK (1996 - 1998)
    uk_holidays = pd.to_datetime([
        '1996-01-01', '1996-04-05', '1996-04-08', '1996-05-06', '1996-05-27', '1996-08-26', '1996-12-25', '1996-12-26',
        '1997-01-01', '1997-03-28', '1997-03-31', '1997-05-05', '1997-05-26', '1997-08-25', '1997-12-25', '1997-12-26',
        '1998-01-01', '1998-04-10', '1998-04-13', '1998-05-04', '1998-05-25', '1998-08-31', '1998-12-25', '1998-12-26'
    ])
    df['is_uk_holiday'] = df['Date'].isin(uk_holidays).astype(int)
    
    # Đếm ngược đến Giáng Sinh (Chỉ kích hoạt trong tháng 12)
    df['days_until_xmas'] = df['Date'].apply(
        lambda x: (pd.to_datetime(f"{x.year}-12-25") - x).days if x.month == 12 and x.day <= 25 else 0
    )
    
    # Đánh dấu mùa Lễ hội (Festive Season) - Bẻ gãy chu kỳ thường
    df['is_festive_season'] = (
        ((df['Date'].dt.month == 12) & (df['Date'].dt.day >= 20)) | 
        ((df['Date'].dt.month == 1) & (df['Date'].dt.day <= 2))
    ).astype(int)
    
    # Ngày vàng mua sắm Boxing Day
    df['is_boxing_day'] = ((df['Date'].dt.month == 12) & (df['Date'].dt.day == 26)).astype(int)

    # =======================================================
    # 2. XỬ LÝ TARGET GỌN GÀNG (Chuẩn bị nguyên liệu cho Lag/Rolling)
    # =======================================================
    df['Withdrawal_clean'] = df.groupby('ATM_ID')['Withdrawal'].ffill()
    if is_train:
        df['Withdrawal_clean'] = df.groupby('ATM_ID')['Withdrawal_clean'].bfill()
        
    # =======================================================
    # 3. LAG FEATURES
    # =======================================================
    lags = [1, 7, 14, 28, 56]
    for lag in lags:
        df[f'lag_{lag}'] = df.groupby('ATM_ID')['Withdrawal_clean'].shift(lag)
    
    # =======================================================
    # 4. ROLLING FEATURES (Bắt buộc phải SHIFT 1 ngày)
    # =======================================================
    windows = [7, 14, 28]
    for window in windows:
        shifted_series = df.groupby('ATM_ID')['Withdrawal_clean'].shift(1)
        
        df[f'rolling_mean_{window}'] = shifted_series.groupby(df['ATM_ID']).transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        )
        df[f'rolling_std_{window}'] = shifted_series.groupby(df['ATM_ID']).transform(
            lambda x: x.rolling(window, min_periods=1).std()
        )
    
    # =======================================================
    # 5. XỬ LÝ NaN VÀ DỌN DẸP
    # =======================================================
    feature_cols = [col for col in df.columns if col not in ['Date', 'ATM_ID', 'Withdrawal', 'Withdrawal_clean']]
    
    # Ffill cho các giá trị rolling/lag
    df[feature_cols] = df.groupby('ATM_ID')[feature_cols].ffill()
    
    # Lấp toàn bộ NaN còn sót lại bằng 0
    df[feature_cols] = df[feature_cols].fillna(0) 
    
    # Xóa cột nháp
    df.drop(columns=['Withdrawal_clean'], inplace=True)
    
    return df

In [44]:
# 1. Gọi hàm tạo feature TRÊN TOÀN BỘ DỮ LIỆU LIỀN MẠCH
print("🔄 Đang tạo features trên toàn bộ trục thời gian liền mạch...")
full_feat_df = create_features(data_with_gt)

# 2. CẮT DỮ LIỆU CƠ BẢN
FORECAST_HORIZON = 56
print("✂️ Đang cắt dữ liệu thành Train/Val/Test...")

test_feat = full_feat_df.groupby('ATM_ID').tail(FORECAST_HORIZON).copy()

val_feat = full_feat_df.groupby('ATM_ID').apply(
    lambda x: x.iloc[-FORECAST_HORIZON*2 : -FORECAST_HORIZON]
).reset_index(drop=True)

# Đây là tập Train gốc (vẫn chứa NaN, giữ nguyên cấu trúc thời gian)
train_feat = full_feat_df.groupby('ATM_ID').apply(
    lambda x: x.iloc[:-FORECAST_HORIZON*2]
).reset_index(drop=True)

print(f"📊 Train shape       : {train_feat.shape}")
print(f"📊 Val shape         : {val_feat.shape}")
print(f"📊 Test (internal)   : {test_feat.shape}")

print(f"\nTrain date     : {train_feat['Date'].min()} → {train_feat['Date'].max()}")
print(f"Val date       : {val_feat['Date'].min()} → {val_feat['Date'].max()}")
print(f"Test date      : {test_feat['Date'].min()} → {test_feat['Date'].max()}")


🔄 Đang tạo features trên toàn bộ trục thời gian liền mạch...
✂️ Đang cắt dữ liệu thành Train/Val/Test...
📊 Train shape       : (69153, 23)
📊 Val shape         : (6216, 23)
📊 Test (internal)   : (6216, 23)

Train date     : 1996-03-18 00:00:00 → 1997-11-30 00:00:00
Val date       : 1997-12-01 00:00:00 → 1998-01-25 00:00:00
Test date      : 1998-01-26 00:00:00 → 1998-03-22 00:00:00


## 5. Data Quality Checks

In [45]:
# Check missing values
print("🔍 Missing Values Analysis")
print(f"{'='*60}")
for name, df in zip(['Train', 'Val', 'Test (internal)', 'Submission'], 
                     [train_feat, val_feat, test_feat, submission_df]):
    total = df.shape[0]
    missing = df['Withdrawal'].isna().sum()
    pct = (missing / total * 100) if total > 0 else 0
    print(f"{name:<15}: {missing:>5} missing ({pct:>6.2f}%)")
print(f"{'='*60}")

# Check duplicates
print("\n🔍 Duplicate Records Analysis")
print(f"{'='*60}")
for name, df in zip(['Train', 'Val', 'Test (internal)'], 
                     [train_feat, val_feat, test_feat]):
    duplicates = df.duplicated(subset=['ATM_ID', 'Date']).sum()
    print(f"{name:<15}: {duplicates} duplicate rows")
print(f"{'='*60}")

# Data type check
print("\n📊 Data Types & Summary Statistics (Train Set)")
print(f"{'='*60}")
print(train_df.dtypes)
print(f"\n{train_df['Withdrawal'].describe()}")

🔍 Missing Values Analysis
Train          :  1517 missing (  2.19%)
Val            :    75 missing (  1.21%)
Test (internal):    81 missing (  1.30%)
Submission     :  6216 missing (100.00%)

🔍 Duplicate Records Analysis
Train          : 0 duplicate rows
Val            : 0 duplicate rows
Test (internal): 0 duplicate rows

📊 Data Types & Summary Statistics (Train Set)
Date          datetime64[ns]
ATM_ID                object
Withdrawal           float64
dtype: object

count    67636.000000
mean        18.252077
std          9.415493
min          0.000000
25%         11.888480
50%         16.465018
75%         22.961599
max         95.962651
Name: Withdrawal, dtype: float64


## 6. Exploratory Data Analysis (EDA) - Plot Generation

In [46]:
def export_eda_plots_separated(train_df, atm_id=None):
    """
    Generate and export 6 separated EDA plots following Storytelling with Data principles.
    Plots saved to report/figures/ folder.
    """
    # Prepare Data
    df_plot = train_df.copy()
    if atm_id:
        series = df_plot[df_plot['ATM_ID'] == atm_id].set_index('Date')['Withdrawal'].dropna()
        context_str = f"(ATM: {atm_id})"
    else:
        series = df_plot.groupby('Date')['Withdrawal'].mean().dropna()
        context_str = "(Network Average)"

    last_date = series.index[-1]
    
    # ---------------------------------------------------------
    # PLOT 1: TREND & ORIGINAL DATA
    # ---------------------------------------------------------
    fig1, ax1 = plt.subplots(figsize=(12, 5))
    
    ax1.plot(series.index, series.values, color=LIGHT_GREY, linewidth=1)
    rolling_trend = series.rolling(window=28, center=True).mean()
    ax1.plot(rolling_trend.index, rolling_trend, color=MAIN_BLUE, linewidth=2.5)
    
    ax1.text(last_date + pd.Timedelta(days=5), series.values[-1], 'Original\nData', 
             color='#a6a6a6', fontsize=11, va='center')
    ax1.text(last_date + pd.Timedelta(days=5), rolling_trend.dropna().values[-1], 'Trend\n(28-day MA)', 
             color=MAIN_BLUE, fontsize=12, fontweight='bold', va='center')
    
    ax1.set_title(f"Despite stable weekly cycles, withdrawal demand plunges during the year-end holidays {context_str}", 
                  fontsize=14, fontweight='bold', color=MAIN_BLUE, loc='left', pad=15)
    ax1.set_ylabel('Withdrawal Volume', fontsize=11)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    
    sns.despine(ax=ax1)
    ax1.yaxis.grid(True, linestyle='-', color='#eeeeee')
    
    plt.tight_layout()
    plt.savefig('../report/figures/01_trend_analysis.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig1)

    # ---------------------------------------------------------
    # PLOT 2: AUTOCORRELATION (ACF)
    # ---------------------------------------------------------
    fig2, ax2 = plt.subplots(figsize=(10, 4))
    
    plot_acf(series, lags=35, ax=ax2, color=MAIN_BLUE, vlines_kwargs={"colors": MAIN_BLUE}, alpha=0.05)
    
    ax2.set_title("Autocorrelation confirms a strict 7-day cyclical pattern in cash demand", 
                  fontsize=14, fontweight='bold', loc='left', pad=15)
    ax2.set_xlabel('Lag (Days)', fontsize=11)
    ax2.set_ylabel('ACF', fontsize=11)
    ax2.set_xticks(range(0, 36, 7))
    
    sns.despine(ax=ax2)
    ax2.grid(axis='y', linestyle='-', color='#eeeeee')
    for lag in range(7, 36, 7):
        ax2.axvline(lag, color=LIGHT_GREY, linestyle='--', linewidth=1, zorder=0)
        
    plt.tight_layout()
    plt.savefig('../report/figures/02_autocorrelation.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig2)

    # ---------------------------------------------------------
    # PLOT 3: PARTIAL AUTOCORRELATION (PACF)
    # ---------------------------------------------------------
    fig3, ax3 = plt.subplots(figsize=(10, 4))
    
    plot_pacf(series, lags=35, ax=ax3, color=MAIN_BLUE, vlines_kwargs={"colors": MAIN_BLUE}, 
              alpha=0.05, method='ywm')
    
    ax3.set_title("Partial Autocorrelation reveals direct dependencies at lag 7", 
                  fontsize=14, fontweight='bold', loc='left', pad=15)
    ax3.set_xlabel('Lag (Days)', fontsize=11)
    ax3.set_ylabel('PACF', fontsize=11)
    ax3.set_xticks(range(0, 36, 7))
    
    sns.despine(ax=ax3)
    ax3.grid(axis='y', linestyle='-', color='#eeeeee')
    for lag in range(7, 36, 7):
        ax3.axvline(lag, color=LIGHT_GREY, linestyle='--', linewidth=1, zorder=0)
        
    plt.tight_layout()
    plt.savefig('../report/figures/03_partial_autocorrelation.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig3)

    # ---------------------------------------------------------
    # PLOT 4: VOLATILITY
    # ---------------------------------------------------------
    fig4, ax4 = plt.subplots(figsize=(12, 4))
    
    vol_short = series.rolling(window=7).std()
    vol_long = series.rolling(window=28).std()
    
    ax4.plot(vol_short.index, vol_short, color=LIGHT_GREY, linewidth=1)
    ax4.plot(vol_long.index, vol_long, color='#e74c3c', linewidth=2)
    
    ax4.text(last_date + pd.Timedelta(days=5), vol_long.dropna().values[-1], 
             'Long-term Volatility\n(Risk Indicator)', 
             color='#e74c3c', fontsize=11, fontweight='bold', va='center')
    
    ax4.set_title(f"Network volatility (cash-out risk) explodes during the transition into the New Year {context_str}", 
                  fontsize=14, fontweight='bold', loc='left', pad=15)
    ax4.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    
    sns.despine(ax=ax4)
    ax4.yaxis.grid(True, linestyle='-', color='#eeeeee')
    
    plt.tight_layout()
    plt.savefig('../report/figures/04_volatility.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig4)

    # ---------------------------------------------------------
    # DATA PREP FOR BOXPLOTS
    # ---------------------------------------------------------
    df_box = series.reset_index()
    df_box['DayOfWeek'] = df_box['Date'].dt.day_name()
    df_box['Month'] = df_box['Date'].dt.month_name()
    days_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    months_order = ['January', 'February', 'March', 'April', 'May', 'June', 
                    'July', 'August', 'September', 'October', 'November', 'December']

    # ---------------------------------------------------------
    # PLOT 5: WEEKLY SEASONALITY
    # ---------------------------------------------------------
    fig5, ax5 = plt.subplots(figsize=(10, 5))
    
    sns.boxplot(data=df_box, y='DayOfWeek', x='Withdrawal', order=days_order, ax=ax5, 
                color='#e6ecf3', orient='h', 
                medianprops={'color': MAIN_BLUE, 'linewidth': 2},
                flierprops={'markerfacecolor': AXIS_GREY, 'markeredgecolor': 'none', 'markersize': 3})
    
    ax5.set_title("Intra-week demand steadily accumulates, peaking on Fridays", 
                  fontsize=14, fontweight='bold', loc='left', pad=15)
    ax5.set_ylabel('')
    ax5.set_xlabel('Withdrawal Volume', fontsize=11)
    
    sns.despine(ax=ax5, left=True)
    ax5.xaxis.grid(True, linestyle='-', color='#eeeeee')
    
    plt.tight_layout()
    plt.savefig('../report/figures/05_weekly_seasonality.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig5)

    # ---------------------------------------------------------
    # PLOT 6: MONTHLY SEASONALITY
    # ---------------------------------------------------------
    fig6, ax6 = plt.subplots(figsize=(10, 6))
    
    sns.boxplot(data=df_box, y='Month', x='Withdrawal', order=months_order, ax=ax6, 
                color='#e6ecf3', orient='h', 
                medianprops={'color': MAIN_BLUE, 'linewidth': 2},
                flierprops={'markerfacecolor': AXIS_GREY, 'markeredgecolor': 'none', 'markersize': 3})
    
    ax6.set_title("December exhibits extreme variance, followed by a sharp drop in January", 
                  fontsize=14, fontweight='bold', loc='left', pad=15)
    ax6.set_ylabel('')
    ax6.set_xlabel('Withdrawal Volume', fontsize=11)
    
    sns.despine(ax=ax6, left=True)
    ax6.xaxis.grid(True, linestyle='-', color='#eeeeee')
    
    plt.tight_layout()
    plt.savefig('../report/figures/06_monthly_seasonality.png', dpi=300, bbox_inches='tight', facecolor='white')
    plt.close(fig6)

    print("✅ Successfully exported 6 separate EDA plots to report/figures/:")
    print("   1. 01_trend_analysis.png")
    print("   2. 02_autocorrelation.png")
    print("   3. 03_partial_autocorrelation.png")
    print("   4. 04_volatility.png")
    print("   5. 05_weekly_seasonality.png")
    print("   6. 06_monthly_seasonality.png")

## 7. Generate EDA Plots

In [47]:
# Generate EDA plots for network average
print("🎨 Generating EDA plots for network average...\n")
export_eda_plots_separated(train_df, atm_id=None)

# Optionally generate plots for first ATM as sample
# sample_atm = train_df['ATM_ID'].unique()[0]
# print(f"\n🎨 Generating EDA plots for sample ATM: {sample_atm}...\n")
# export_eda_plots_separated(train_df, atm_id=sample_atm)

🎨 Generating EDA plots for network average...

✅ Successfully exported 6 separate EDA plots to report/figures/:
   1. 01_trend_analysis.png
   2. 02_autocorrelation.png
   3. 03_partial_autocorrelation.png
   4. 04_volatility.png
   5. 05_weekly_seasonality.png
   6. 06_monthly_seasonality.png


## 8. Save Processed Data

In [48]:
# Save processed datasets
print("💾 Saving processed datasets to data/processed/...\n")

# Save as CSV files (standard format)
train_feat.to_csv('../data/processed/train_feat.csv', index=False)
val_feat.to_csv('../data/processed/val_feat.csv', index=False)
test_feat.to_csv('../data/processed/test_feat.csv', index=False)
submission_df.to_csv('../data/processed/submission_df.csv', index=False)

# Also save as pickle files (preserve dtypes)
train_feat.to_pickle('../data/processed/train_feat.pkl')
val_feat.to_pickle('../data/processed/val_feat.pkl')
test_feat.to_pickle('../data/processed/test_feat.pkl')
submission_df.to_pickle('../data/processed/submission_df.pkl')

print("✅ Successfully saved datasets:")
print("   - data/processed/train_feat.csv & .pkl")
print("   - data/processed/val_feat.csv & .pkl")
print("   - data/processed/test_feat.csv & .pkl")
print("   - data/processed/submission_df.csv & .pkl")

print(f"\n📊 Final Summary:")
print(f"{'='*60}")
print(f"Train:      {train_feat.shape[0]:>8} rows | {train_feat['ATM_ID'].nunique()} ATMs")
print(f"Val:        {val_feat.shape[0]:>8} rows | {val_feat['ATM_ID'].nunique()} ATMs")
print(f"Test:       {test_feat.shape[0]:>8} rows | {test_feat['ATM_ID'].nunique()} ATMs")
print(f"Submission: {submission_df.shape[0]:>8} rows | {submission_df['ATM_ID'].nunique()} ATMs")
print(f"{'='*60}")

💾 Saving processed datasets to data/processed/...

✅ Successfully saved datasets:
   - data/processed/train_feat.csv & .pkl
   - data/processed/val_feat.csv & .pkl
   - data/processed/test_feat.csv & .pkl
   - data/processed/submission_df.csv & .pkl

📊 Final Summary:
Train:         69153 rows | 111 ATMs
Val:            6216 rows | 111 ATMs
Test:           6216 rows | 111 ATMs
Submission:     6216 rows | 111 ATMs
